
This Python notebook run the basic analysis used for the TSP manuscript, it use the data formated and combined trhough notebook A0
It saves a .csv directly usable for GLMM R script

__author__ = "C.Doussot"
__date__ =  18-February-2025

# Prepare the GLMM dataset

Builds one row for every trial, merges questionnaire data, and writes ../Combined_data/GLMM_df_paper.csv. Run this before fitting models in R; the performance notebook consumes the R outputs.

Run from top to bottom in a fresh kernel. Helpers are imported from `a1_helpers.py`.

In [1]:
# Data handling
import os
import re
import csv
import warnings
import itertools as it

import numpy as np
import pandas as pd
from tqdm import tqdm

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.patches import FancyArrowPatch

# Networks and statistical tests
import networkx as nx
from scipy.stats import binomtest, chi2_contingency, chisquare
import statsmodels.stats.multitest as stm

# Functions stored in a1_helpers.py
from a1_helpers import (
    get_time_taken_sec,
    extract_strategies,
    get_percentages,
    parse_values,
    directed_to_undirected_with_summed_weights,
    graph_primer,
    create_dic,
    percentage_scaler,
    get_transition_df,
    Euclidean_Dist,
)

# Progress bars for pandas
tqdm.pandas()

# Hide deprecation warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
from pathlib import Path

# Run with the notebook's Analysis folder as the working directory.
assert Path('a1_helpers.py').is_file(), 'Set the working directory to the Analysis folder.'
fig_folder = '../Figure_paper/'
Path(fig_folder).mkdir(parents=True, exist_ok=True)
colors = ['#0033ff', '#ff4ff1']

In [3]:
sequence_df = pd.read_csv('../Combined_data/seq_df_2025.csv', index_col=0)
sequence_df['game'] = sequence_df['experiment'].str[-1]

In [4]:
# Load the questionnaire data processed by G_results_questionnaire.
participants_details = pd.read_csv(
    '../Combined_data/Participants_details.csv',
    index_col=0,
)

# Remove records without a participant ID.
participants_details = participants_details.dropna(subset=['ppid'])

# Add one questionnaire record to each participant's sequence rows.
sequence_df = sequence_df.merge(
    participants_details,
    on='ppid',
    how='left',
    validate='many_to_one',
    indicator=True,
)

# Report participants whose questionnaire data are missing.
missing = sequence_df.loc[
    sequence_df['_merge'] == 'left_only', 'ppid'
].unique()

print("Participants without questionnaire data:", missing.tolist())

sequence_df = sequence_df.drop(columns='_merge')

Participants without questionnaire data: ['Dolmen', 'Morne']


In [5]:
# Keep EVERY trial; repeated node records within each trial become one row.
toplot = sequence_df.groupby(
    ['ppid', 'cond', 'scale', 'trial_num']
).first().reset_index()

In [6]:
toplot['time_seconds'] = pd.to_timedelta(toplot['time_taken'], errors='raise').dt.total_seconds()
# Restore the original A1 definition: subtract each participant's best error
# within each condition, across all scales and trials (percentage points).
for metric in ['error_traj', 'error_seq']:
    best = toplot.groupby(['ppid', 'cond'])[metric].transform('min')
    toplot[f'norm_{metric}'] = toplot[metric] - best

In [7]:
toplot = toplot.loc[:, ['ppid', 'trial_num', 'scale', 'cond', 'experiment', 'session_num','error_seq','error_traj','TSP_correct','difficulty', 'self_assessment', 'best_estimated_scale','Age', 'Gender', 'gaming_habits',
       'navigation_rating', 'GPS_uage',
       'daily_optimisation', 'norm_error_traj',
       'norm_error_seq','time_seconds']]

In [8]:
toplot.to_csv('../Combined_data/GLMM_df_paper.csv')